# PBIT Safety — Validation Methodology Study

Four sub-experiments that examine the robustness of the cross-validation setup.

| Sub-exp | Question | File |
|---------|----------|------|
| **C1** | Leakage: grouped CV vs random stratified CV — how big is the optimism gap? | `cv_leakage_comparison.csv` |
| **C2** | F06 sensitivity: removing each F06 serial number from training, one at a time | `cv_composite6_sensitivity.csv` |
| **C3** | Stratification strategy: Fault_Mode vs Severity_Index vs ungrouped | `cv_stratification_comparison.csv` |
| **C4** | Seed stability: 10 different seeds — do conclusions depend on fold randomness? | `cv_seed_stability.csv` |

**Fixed model**: LightGBM, F1-tuned params, Aug + Freq×Sev (current scale), f05_boost=6.98×.

## Setup

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks_safety':
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold

from pbit.modeling import (
    META_COLS, FAULT_ORDER, SEVERITY_MAP,
    make_cv_folds, summarize_cv,
    compute_sample_weights, evaluate_predictions, compute_safety_metrics,
)
from pbit.data_validation import load_long_dataset
from pbit.augmentation import augment_training_set
from pbit.features import extract_features

TRAIN_PATH   = PROJECT_ROOT / 'data' / 'train_real.csv'
LONG_PATH    = PROJECT_ROOT / 'data' / 'PBIT_atuador_DESCARAC_r1_long_clean.csv'
F1_PARAMS    = PROJECT_ROOT / 'data' / 'best_lgbm_params.json'

PROJECT_ROOT

## Load Data & Model Params

In [ ]:
train_df  = pd.read_csv(TRAIN_PATH)
long_df   = load_long_dataset(LONG_PATH)
feat_cols = [c for c in train_df.columns if c not in META_COLS]

with open(F1_PARAMS) as fh:
    f1_all = json.load(fh)
f05_boost  = f1_all.pop('f05_boost')
lgb_params = f1_all

print(f'Training set : {len(train_df):,} experiments x {len(feat_cols)} features')
print(f'f05_boost    : {f05_boost:.4f}')
print(f'F06 serial numbers in train: {sorted(train_df[train_df.Fault_Mode=="F06"].Serial_Number.unique())}')

## Shared CV Helper

In [ ]:
def run_cv_folds(folds, label='', aug=True, weight_mode='freq_sev'):
    """Run 5 folds and return summarized metrics dict + raw safety lists."""
    fold_results, fold_cmrs, fold_down, fold_severr = [], [], [], []

    for fold_i, (tr_idx, val_idx) in enumerate(folds):
        tr_real  = train_df.iloc[tr_idx]
        val_real = train_df.iloc[val_idx]

        if aug:
            tr_exp_ids = set(tr_real['Experiment_Index'].tolist())
            aug_long   = augment_training_set(
                long_df, tr_exp_ids,
                target_per_class=160, noise_scale=0.01,
                random_state=42 + fold_i,
            )
            if len(aug_long) > 0:
                aug_feat = extract_features(aug_long)
                tr_fold  = pd.concat([tr_real, aug_feat], ignore_index=True)
            else:
                tr_fold = tr_real
        else:
            tr_fold = tr_real

        X_train = tr_fold[feat_cols].values.astype(float)
        y_train = tr_fold['Fault_Mode'].values
        X_val   = val_real[feat_cols].values.astype(float)
        y_val   = val_real['Fault_Mode'].values

        sw = compute_sample_weights(y_train, weight_mode)
        sw[y_train == 'F05'] *= f05_boost

        model = lgb.LGBMClassifier(**lgb_params)
        model.fit(X_train, y_train, sample_weight=sw)
        y_pred = model.predict(X_val)

        m  = evaluate_predictions(y_val, y_pred)
        sm = compute_safety_metrics(y_val, y_pred)
        m['fold'] = fold_i
        fold_results.append(m)
        fold_cmrs.append(sm['cmr'])
        fold_down.append(sm['down_risk'])
        fold_severr.append(sm['sev_err'])

    s = summarize_cv(fold_results)
    s['cmr_cv']    = float(np.nanmean([v for v in fold_cmrs if not np.isnan(v)] or [float('nan')]))
    s['down_cv']   = float(np.mean(fold_down))
    s['severr_cv'] = float(np.mean(fold_severr))
    return s


print('Helper defined.')

---
## C1 — Leakage Quantification

Compare grouped CV (StratifiedGroupKFold by Serial_Number) vs. random stratified CV (StratifiedKFold).
Same model, same config, same seed. The gap quantifies the optimism introduced when the same
equipment appears in both training and validation.

In [ ]:
# Grouped CV — standard (no leakage)
grouped_folds = make_cv_folds(train_df, n_splits=5, random_state=42)

# Random stratified CV — allows same equipment in train and val (optimistic)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
random_folds = list(skf.split(train_df, train_df['Fault_Mode']))

print('Running grouped CV...')
t0 = time.time()
s_grouped = run_cv_folds(grouped_folds)
print(f'  Done in {time.time()-t0:.0f}s')

print('Running random stratified CV...')
t0 = time.time()
s_random = run_cv_folds(random_folds)
print(f'  Done in {time.time()-t0:.0f}s')

print()
print('=== C1 — Leakage quantification ===')
print(f"{'Metric':<25} {'Grouped':>10} {'Random':>10} {'Gap':>10}")
print('-' * 55)
for label, k in [
    ('Macro F1',    'macro_f1_mean'),
    ('Balanced Acc','balanced_accuracy_mean'),
    ('F06 Recall',  'f06_recall_mean'),
    ('F05 Recall',  'f05_recall_mean'),
    ('F04 Recall',  'f04_recall_mean'),
    ('Sev3 Recall', 'sev3_recall_mean'),
]:
    vg = s_grouped.get(k, float('nan'))
    vr = s_random.get(k, float('nan'))
    print(f"{label:<25} {vg:>10.4f} {vr:>10.4f} {vr-vg:>+10.4f}")
print()
for label, kg, kr in [
    ('CMR',           'cmr_cv',    'cmr_cv'),
    ('Downward Risk', 'down_cv',   'down_cv'),
    ('Severity Error','severr_cv', 'severr_cv'),
]:
    vg = s_grouped.get(kg, float('nan'))
    vr = s_random.get(kr, float('nan'))
    print(f"{label:<25} {vg:>10.4f} {vr:>10.4f} {vr-vg:>+10.4f}")

# Save
c1_df = pd.DataFrame([
    {'Method': 'Grouped (no leakage)', **{k: s_grouped.get(k, float('nan'))
        for k in ['macro_f1_mean','balanced_accuracy_mean','f06_recall_mean',
                  'f04_recall_mean','sev3_recall_mean','cmr_cv','down_cv','severr_cv']}},
    {'Method': 'Random stratified',    **{k: s_random.get(k, float('nan'))
        for k in ['macro_f1_mean','balanced_accuracy_mean','f06_recall_mean',
                  'f04_recall_mean','sev3_recall_mean','cmr_cv','down_cv','severr_cv']}},
])
c1_df.to_csv(PROJECT_ROOT / 'data' / 'cv_leakage_comparison.csv', index=False)
print('\nSaved: data/cv_leakage_comparison.csv')

---
## C2 — F06 Sensitivity (Leave-One-SN-Out)

F06 has only 6 training serial numbers (SN 045, 054, 056 — from the train split).  
Each run removes one F06 serial number from training entirely, trains on the rest, and reports
F06 recall and CMR on the standard 5-fold CV. This quantifies how much each SN contributes
to F06 recognition.

In [ ]:
f06_sns = sorted(train_df[train_df['Fault_Mode'] == 'F06']['Serial_Number'].unique())
print(f'F06 serial numbers in training set: {f06_sns}')
print(f'F06 experiments per SN:')
print(train_df[train_df['Fault_Mode'] == 'F06'].groupby('Serial_Number').size().to_string())

c2_rows = []

# Baseline: all F06 SNs present
baseline_folds = make_cv_folds(train_df, n_splits=5, random_state=42)
print('\nRunning C2 baseline (all F06 SNs)...')
s_base = run_cv_folds(baseline_folds)
c2_rows.append({
    'Config': 'All F06 SNs',
    'Removed_SN': 'none',
    'F06_train_n': len(train_df[train_df['Fault_Mode'] == 'F06']),
    'CMR': round(s_base['cmr_cv'], 4),
    'F06_Recall': round(s_base['f06_recall_mean'], 4),
    'Down_Risk': round(s_base['down_cv'], 4),
    'Macro_F1': round(s_base['macro_f1_mean'], 4),
})
print(f'  CMR={s_base["cmr_cv"]:.4f}  F06={s_base["f06_recall_mean"]:.4f}  F1={s_base["macro_f1_mean"]:.4f}')

for sn in f06_sns:
    mask = ~((train_df['Fault_Mode'] == 'F06') & (train_df['Serial_Number'] == sn))
    reduced_df = train_df[mask].reset_index(drop=True)
    n_f06 = len(reduced_df[reduced_df['Fault_Mode'] == 'F06'])

    # Rebuild folds on reduced dataset
    sgkf = StratifiedGroupKFold(n_splits=5)
    red_folds = list(sgkf.split(
        reduced_df,
        reduced_df['Fault_Mode'],
        groups=reduced_df['Serial_Number'],
    ))

    # Temporarily swap train_df for the reduced version
    orig_train_df = train_df
    import builtins
    _train_df_backup = train_df

    # Run CV manually on reduced_df
    fold_results, fold_cmrs, fold_down = [], [], []
    for fold_i, (tr_idx, val_idx) in enumerate(red_folds):
        tr_real  = reduced_df.iloc[tr_idx]
        val_real = reduced_df.iloc[val_idx]
        tr_exp_ids = set(tr_real['Experiment_Index'].tolist())
        aug_long   = augment_training_set(
            long_df, tr_exp_ids,
            target_per_class=160, noise_scale=0.01, random_state=42 + fold_i,
        )
        if len(aug_long) > 0:
            aug_feat = extract_features(aug_long)
            tr_fold  = pd.concat([tr_real, aug_feat], ignore_index=True)
        else:
            tr_fold = tr_real

        X_train = tr_fold[feat_cols].values.astype(float)
        y_train = tr_fold['Fault_Mode'].values
        X_val   = val_real[feat_cols].values.astype(float)
        y_val   = val_real['Fault_Mode'].values

        sw = compute_sample_weights(y_train, 'freq_sev')
        sw[y_train == 'F05'] *= f05_boost

        model = lgb.LGBMClassifier(**lgb_params)
        model.fit(X_train, y_train, sample_weight=sw)
        y_pred = model.predict(X_val)

        m  = evaluate_predictions(y_val, y_pred)
        sm = compute_safety_metrics(y_val, y_pred)
        m['fold'] = fold_i
        fold_results.append(m)
        fold_cmrs.append(sm['cmr'])
        fold_down.append(sm['down_risk'])

    s = summarize_cv(fold_results)
    cmr_mean  = float(np.nanmean([v for v in fold_cmrs if not np.isnan(v)] or [float('nan')]))
    down_mean = float(np.mean(fold_down))

    c2_rows.append({
        'Config': f'Remove SN {sn}',
        'Removed_SN': sn,
        'F06_train_n': n_f06,
        'CMR': round(cmr_mean, 4),
        'F06_Recall': round(s['f06_recall_mean'], 4),
        'Down_Risk': round(down_mean, 4),
        'Macro_F1': round(s['macro_f1_mean'], 4),
    })
    print(f'  Remove SN {sn}  (F06 n={n_f06})  CMR={cmr_mean:.4f}  F06={s["f06_recall_mean"]:.4f}  F1={s["macro_f1_mean"]:.4f}')

c2_df = pd.DataFrame(c2_rows)
c2_df.to_csv(PROJECT_ROOT / 'data' / 'cv_composite6_sensitivity.csv', index=False)
print('\nSaved: data/cv_composite6_sensitivity.csv')
c2_df

---
## C3 — Stratification Strategy

Compare three fold-generation strategies:
- **A** (current): StratifiedGroupKFold, stratify by `Fault_Mode`, group by `Serial_Number`
- **B**: StratifiedGroupKFold, stratify by `Severity_Index`, group by `Serial_Number`
- **C**: StratifiedKFold only by `Fault_Mode` — no grouping (allows same equipment in train+val)

In [ ]:
# Strategy A — current (grouped by SN, stratified by Fault_Mode)
folds_A = make_cv_folds(train_df, n_splits=5, random_state=42)

# Strategy B — grouped by SN, stratified by Severity_Index
sev_labels = train_df['Fault_Mode'].map(SEVERITY_MAP).astype(str)
sgkf_b = StratifiedGroupKFold(n_splits=5)
folds_B = list(sgkf_b.split(train_df, sev_labels, groups=train_df['Serial_Number']))

# Strategy C — ungrouped stratified by Fault_Mode
skf_c = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds_C = list(skf_c.split(train_df, train_df['Fault_Mode']))

c3_rows = []
for name, folds in [('A: Grouped / FaultMode', folds_A),
                    ('B: Grouped / Severity',  folds_B),
                    ('C: Ungrouped / FaultMode', folds_C)]:
    print(f'Running {name}...')
    t0 = time.time()
    s = run_cv_folds(folds)
    print(f'  CMR={s["cmr_cv"]:.4f}  F1={s["macro_f1_mean"]:.4f}  F06={s["f06_recall_mean"]:.4f}  ({time.time()-t0:.0f}s)')
    c3_rows.append({
        'Strategy': name,
        'CMR': round(s['cmr_cv'], 4),
        'Down_Risk': round(s['down_cv'], 4),
        'Macro_F1': round(s['macro_f1_mean'], 4),
        'Bal_Acc': round(s['balanced_accuracy_mean'], 4),
        'F06_Recall': round(s['f06_recall_mean'], 4),
        'F04_Recall': round(s['f04_recall_mean'], 4),
        'Sev3_Recall': round(s['sev3_recall_mean'], 4),
    })

c3_df = pd.DataFrame(c3_rows)
c3_df.to_csv(PROJECT_ROOT / 'data' / 'cv_stratification_comparison.csv', index=False)
print('\nSaved: data/cv_stratification_comparison.csv')
print()
print('=== C3 — Stratification strategy comparison ===')
print(c3_df.to_string(index=False))

---
## C4 — Seed Stability

10 different random seeds for fold generation (grouped StratifiedGroupKFold).  
Same model and config throughout. Low std confirms conclusions are not seed-dependent.

Seeds: 0, 7, 13, 21, 42, 55, 77, 100, 123, 200.

In [ ]:
SEEDS = [0, 7, 13, 21, 42, 55, 77, 100, 123, 200]
c4_rows = []

t0_total = time.time()
for seed in SEEDS:
    folds_s = make_cv_folds(train_df, n_splits=5, random_state=seed)
    s = run_cv_folds(folds_s)
    c4_rows.append({
        'Seed': seed,
        'CMR': round(s['cmr_cv'], 4),
        'Down_Risk': round(s['down_cv'], 4),
        'Sev_Err': round(s['severr_cv'], 4),
        'Macro_F1': round(s['macro_f1_mean'], 4),
        'Bal_Acc': round(s['balanced_accuracy_mean'], 4),
        'F06_Recall': round(s['f06_recall_mean'], 4),
        'F05_Recall': round(s['f05_recall_mean'], 4),
        'Sev3_Recall': round(s['sev3_recall_mean'], 4),
    })
    print(f'  seed={seed:>3}  CMR={s["cmr_cv"]:.4f}  F1={s["macro_f1_mean"]:.4f}  '
          f'F06={s["f06_recall_mean"]:.4f}  DR={s["down_cv"]:.4f}')

print(f'\nTotal time: {(time.time()-t0_total)/60:.1f} min')

c4_df = pd.DataFrame(c4_rows)
c4_df.to_csv(PROJECT_ROOT / 'data' / 'cv_seed_stability.csv', index=False)
print('Saved: data/cv_seed_stability.csv')

### C4 — Summary Statistics

In [ ]:
metrics = ['CMR', 'Down_Risk', 'Macro_F1', 'Bal_Acc', 'F06_Recall', 'Sev3_Recall']

print('=== C4 — Seed stability summary (10 seeds) ===')
print(f"{'Metric':<15} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8} {'CV%':>8}")
print('-' * 58)
for m in metrics:
    vals = c4_df[m]
    mean, std = vals.mean(), vals.std()
    cv_pct = 100 * std / mean if mean != 0 else float('nan')
    print(f"{m:<15} {mean:>8.4f} {std:>8.4f} {vals.min():>8.4f} {vals.max():>8.4f} {cv_pct:>7.1f}%")

print()
# Flag if any metric has CV% > 10% — indicates instability
for m in metrics:
    vals = c4_df[m]
    mean, std = vals.mean(), vals.std()
    cv_pct = 100 * std / mean if mean != 0 else float('nan')
    if cv_pct > 10:
        print(f'  WARNING: {m} has high variability (CV={cv_pct:.1f}%) — conclusions may be seed-dependent')
    else:
        print(f'  OK: {m} is stable (CV={cv_pct:.1f}%)')

### C4 — Stability Plot

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, m in zip(axes, metrics):
    vals = c4_df[m].values
    mean, std = vals.mean(), vals.std()
    ax.bar(range(len(SEEDS)), vals, color='steelblue', alpha=0.8, edgecolor='white')
    ax.axhline(mean, color='tomato', linewidth=1.5, linestyle='--', label=f'Mean={mean:.4f}')
    ax.axhspan(mean - std, mean + std, alpha=0.15, color='tomato', label=f'±1σ={std:.4f}')
    ax.set_xticks(range(len(SEEDS)))
    ax.set_xticklabels([str(s) for s in SEEDS], fontsize=8)
    ax.set_xlabel('Random seed')
    ax.set_title(m)
    ax.legend(fontsize=8)
    ax.grid(True, axis='y', linewidth=0.3)

fig.suptitle('C4 — Seed stability: grouped 5-fold CV across 10 seeds\n'
             'LightGBM F1-tuned params, Aug + Freq×Sev', fontsize=11)
plt.tight_layout()
plt.show()

---
## Summary

In [ ]:
print('=== Validation Study — Summary ===')
print()
print('C1 — Leakage gap (Random − Grouped):')
if len(c1_df) == 2:
    gr = c1_df[c1_df.Method.str.contains('Grouped')].iloc[0]
    ra = c1_df[c1_df.Method.str.contains('Random')].iloc[0]
    for col in ['macro_f1_mean', 'f06_recall_mean', 'cmr_cv']:
        print(f'  {col:<25}: grouped={gr[col]:.4f}  random={ra[col]:.4f}  gap={ra[col]-gr[col]:+.4f}')

print()
print('C2 — F06 sensitivity (F06 recall range):')
print(f'  Baseline: {c2_df[c2_df.Removed_SN=="none"]["F06_Recall"].values[0]:.4f}')
excl = c2_df[c2_df.Removed_SN != 'none']
print(f'  With one SN removed: {excl["F06_Recall"].min():.4f} – {excl["F06_Recall"].max():.4f}')

print()
print('C3 — Stratification strategy (CMR / Macro F1):')
for _, row in c3_df.iterrows():
    print(f'  {row["Strategy"]:<35}: CMR={row["CMR"]:.4f}  F1={row["Macro_F1"]:.4f}')

print()
print('C4 — Seed stability (mean ± std):')
for m in ['CMR', 'Macro_F1', 'F06_Recall']:
    mean, std = c4_df[m].mean(), c4_df[m].std()
    print(f'  {m:<15}: {mean:.4f} ± {std:.4f}  (CV={100*std/mean:.1f}%)')